Cài đặt streamlit

In [7]:
pip install -q streamlit pypdf chromadb ollama

import và cấu hình

In [8]:
import streamlit as st
import tempfile, os, time
import pypdf
import chromadb
import ollama

#tên model gán thành biến để sau này dễ đổi
LLM_MODEL ="vicuna:7b-v1.5-q5_1"
EMBEB_MODEL ="bge-m3"

PROMPT = """Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
11 Nếu ngữ cảnh không có thông tin, hãy nói là bạn không biết, đừng bịa.
12 Trả lời ngắn gọn, chính xác, bằng tiếng Việt.
13
14 Ngữ cảnh: {context}
15
16 Câu hỏi: {question}
17 Trả lời:"""

khởi tạo session state

In [10]:
#ta muốn lưu dữ liệu lại sau mỗi lần tương tác với web: giữ lại vector database, tên file đang xử lý và lịch sử chat
for k, v in {"collection": None, "pdf_name": "", "chat_history": []}.items():
 st.session_state.setdefault(k,v)

2026-06-26 18:45:38.211 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 18:45:38.212 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-06-26 18:45:38.213 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 18:45:38.214 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 18:45:38.215 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 18:45:38.216 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

các hàm sử lý

In [11]:
def embed(texts):
  #chuyển text thành vector embedding
  return ollama.embed(model=EMBEB_MODEL, input = texts)["embeddings"]

def chunk_text(text, size = 1000, overlap=200):
  #cắt text thành các chunk nhỏ
  paras = [p.strip() for p in text.split("\n") if p.strip()]
  chunks, cur = [], ""
  for p in paras:
    if len(cur) + len(p) + 1 <= size:
      cur += p + "\n"
    else:
      if cur:
        chunks.append(cur.strip())
      cur = (cur[-overlap:] + p + "\n") if overlap else (p + "\n")
  if cur.strip():
    chunks.append(cur.strip())
  return chunks

def process_pdf(uploaded_file):
  #đọc file, cắt nhỏ, embedding và lưu cào chromaDB
  #lưu upload file thành file tạm
  with tempfile.NameTemporaryFile(delete = False, suffix = ".pdf") as tmp:
    tmp.write(uploaded_file.getvalue())
    path = tmp.name

  #đọc nội dung file
  text ="\n".join(p.extract_text() or " for p in pypdf.PdfReader(path).pages")
  os.unlink(path) #xóa file tạm

  #cắt nhỏ và lưu vàu chromaDB
  chunks = chunk_text(text)
  client = chromadb.Client()
  col = client.get_ỏ_create_collection(f"rag_{int(time.time())}")
  col.add(
      ids=[str(i) for i in range(len(chunks))],
      documents = chunks,
      embeddings = embed(chunks)
  )
  return col, len(chunks)

def rag(question, collection, k=4):
  #hàm RAG: tìm context và hỏi LLM
  res = collection.query(query_embeddings = embed([question]), n_results=k)
  context = "\n\n".join(res["document"][0])
  resp = ollama.chat(
      model = LLM+MODEL,
      messages = [{"role": "user", "content": PROMPT.format(context=context, question=question)}]
  )
  return resp["message"]["content"]